In [ ]:
import pandas as pd

In [ ]:
file = '/home/maciej/PycharmProjects/NYacc/data/ny_collisions.csv'

In [ ]:
df = pd.read_csv(file, dtype={'ZIP CODE': str}, parse_dates=['ACCIDENT DATE'], keep_default_na=False, na_values=[''])

In [ ]:
df.dtypes

In [ ]:
df.to_parquet('ny_collisions.parquet', engine='pyarrow', compression='snappy')

In [ ]:
df = pd.read_parquet('ny_collisions.parquet')

In [ ]:
df.dtypes

In [ ]:
df.describe()

Sprawdzenie ilości "pustych" dzielnic

In [ ]:
print(df.groupby('BOROUGH', dropna=False).size())

uzupełnienie brakujących dzielnic z użyciem istniejących danych geolokalizacyjnych

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

ma_dzielnice = df[df['BOROUGH'].notna() & df['LATITUDE'].notna() & df['LONGITUDE'].notna()]
brak_dzielnicy = df[df['BOROUGH'].isna() & df['LATITUDE'].notna() & df['LONGITUDE'].notna()]

if not brak_dzielnicy.empty and not ma_dzielnice.empty:

    knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')

    knn.fit(ma_dzielnice[['LATITUDE', 'LONGITUDE']], ma_dzielnice['BOROUGH'])

    uzupelnione_dzielnice = knn.predict(brak_dzielnicy[['LATITUDE', 'LONGITUDE']])

    df.loc[brak_dzielnicy.index, 'BOROUGH'] = uzupelnione_dzielnice
    print(f"Uzupełniono {len(brak_dzielnicy)} wierszy na podstawie współrzędnych.")
else:
    print("Brak wierszy do uzupełnienia.")

In [ ]:
#df[df.isna().any(axis=1)]

In [ ]:
print(df.groupby('BOROUGH', dropna=False).size())

Uzupełnianie brakujących wartości

In [ ]:
df['GEOM_MISSING'] = df['LATITUDE'].isna().astype(int)

injury_cols = [
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED'
]
df[injury_cols] = df[injury_cols].fillna(0).astype(int)

other_columns = [
    'BOROUGH', 'ZIP CODE',
    'CONTRIBUTING FACTOR VEHICLE 1', 'CONTRIBUTING FACTOR VEHICLE 2', 'CONTRIBUTING FACTOR VEHICLE 3', 'CONTRIBUTING FACTOR VEHICLE 4', 'CONTRIBUTING FACTOR VEHICLE 5',
    'VEHICLE TYPE CODE 1', 'VEHICLE TYPE CODE 2', 'VEHICLE TYPE CODE 3', 'VEHICLE TYPE CODE 4', 'VEHICLE TYPE CODE 5',
    'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME'

]


df[other_columns] = df[other_columns].fillna('UnKnOwN')

In [ ]:
df.to_csv('ny_collisions_updated.csv')

In [ ]:
df.dtypes

In [ ]:
from charts import acc_group_borough
acc_group_borough(df)

In [ ]:
wypadki_dzielnice = df.groupby(['BOROUGH', 'CONTRIBUTING FACTOR VEHICLE 1']).size().reset_index(name = 'liczba wypadków')
wypadki_dzielnice = wypadki_dzielnice[wypadki_dzielnice['BOROUGH'] != 'UnKnOwN']
wypadki_dzielnice = wypadki_dzielnice[wypadki_dzielnice['CONTRIBUTING FACTOR VEHICLE 1'] != 'Unspecified']
print(wypadki_dzielnice)

In [ ]:
wypadki_dzielnice = wypadki_dzielnice.sort_values(by= ['BOROUGH', 'liczba wypadków'], ascending = [True, False])
#print(wypadki_dzielnice)
top3_per_borough = wypadki_dzielnice.groupby('BOROUGH').head(3)
top3_per_borough

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(14, 7))
sns.barplot(
    data=top3_per_borough,
    x="BOROUGH",
    y="liczba wypadków",
    hue="CONTRIBUTING FACTOR VEHICLE 1",
)
plt.xticks(rotation=45)
plt.show()

In [ ]:
from map import accident_map
dzielnica = 'STATEN ISLAND'
df_filtered = df[df['BOROUGH'] == dzielnica]
fig = accident_map(df_filtered)
fig.show()

In [ ]:
df_filtered.head(5)